In [ ]:
# ============================================================
# BLOCK 1 — Imports and local library loading
# ============================================================

%gui qt

from pathlib import Path
import sys
import gc
import pickle
import json
import importlib
import warnings

import numpy as np
import pandas as pd
import tifffile
import napari

from pandas.errors import PerformanceWarning

# ------------------------------------------------------------
# User project folder containing liver_graph_generator.py
# ------------------------------------------------------------

PROJECT_CODE_DIR = Path.home() / "Desktop/Master/thesis/Liver_Tissue/Regeneration"

sys.path.insert(0, str(PROJECT_CODE_DIR))

import liver_graph_generator
importlib.reload(liver_graph_generator)

from liver_graph_generator import (
    LiverGraphGenerator,
    GraphGeneratorConfig,
    FeatureRegistry,
    get_tiff_metadata,
    sanitize_crop_bounds,
)

# ------------------------------------------------------------
# Clean output
# ------------------------------------------------------------

warnings.filterwarnings("ignore", category=PerformanceWarning)

print("Library loaded from:", liver_graph_generator.__file__)
print("Napari ready.")

In [ ]:
# ============================================================
# BLOCK 2 — Utility functions for paths, loading, saving, and meshing
# ============================================================

# ------------------------------------------------------------
# Expected file names inside each sample folder
# ------------------------------------------------------------

EXPECTED_FILES = {
    "cells": "Cells.tiff",
    "nuclei": "Nucleos.tiff",
    "bc": "BC.tiff",
    "sinusoids": "Sinusoids.tiff",
    "vcvp": "CV-PV.tiff",
}


# ------------------------------------------------------------
# Path helpers
# ------------------------------------------------------------

def get_sample_dir(sample_name):
    """
    Return the input folder for one sample.
    """
    return BASE_DATA_DIR / sample_name


def get_sample_output_dir(sample_name):
    """
    Return the output graph folder for one sample.
    """
    return GRAPH_OUTPUT_DIR / sample_name


def get_visualization_cache_dir(sample_name):
    """
    Return the visualization cache folder for one sample.
    """
    return get_sample_output_dir(sample_name) / "visualization_cache"


def get_full_result_path(sample_name):
    """
    Return the path where the full GraphGeneratorResult is stored.
    """
    return get_sample_output_dir(sample_name) / "result_full.pkl"


def get_sample_paths(sample_name):
    """
    Return all expected TIFF paths for one sample.
    """
    sample_dir = get_sample_dir(sample_name)

    paths = {
        key: sample_dir / filename
        for key, filename in EXPECTED_FILES.items()
    }

    missing = [
        str(path)
        for path in paths.values()
        if not path.exists()
    ]

    if len(missing) > 0:
        raise FileNotFoundError(
            "Missing files for sample "
            f"'{sample_name}':\n" + "\n".join(missing)
        )

    return paths


# ------------------------------------------------------------
# TIFF loading helpers
# ------------------------------------------------------------

def open_sample_memmaps(sample_name):
    """
    Open full TIFF images as memmaps.

    This avoids loading every full image eagerly into RAM.
    Napari can use these arrays as image/label sources.
    """
    paths = get_sample_paths(sample_name)

    arrays = {
        key: tifffile.memmap(path)
        for key, path in paths.items()
    }

    return arrays


def inspect_sample_files(sample_name):
    """
    Inspect TIFF metadata for one sample without loading the full data.
    """
    paths = get_sample_paths(sample_name)

    rows = []

    for key, path in paths.items():
        meta = get_tiff_metadata(path)

        rows.append({
            "sample": sample_name,
            "channel": key,
            "path": str(path),
            "shape": meta["shape"],
            "dtype": meta["dtype"],
            "file_size_GiB": meta["file_size_gib"],
            "estimated_RAM_GiB": meta["estimated_ram_gib"],
        })

    df = pd.DataFrame(rows)
    display(df)

    shapes = df["shape"].tolist()
    if not all(shape == shapes[0] for shape in shapes):
        raise ValueError(f"Not all TIFFs have the same shape for sample {sample_name}.")

    return df


# ------------------------------------------------------------
# Result saving/loading
# ------------------------------------------------------------

def save_full_result(result, sample_name):
    """
    Save the full GraphGeneratorResult object.

    This preserves meshes, cell_data, nucleus_data, diagnostics,
    node features, edge features, and graph.
    """
    output_dir = get_sample_output_dir(sample_name)
    output_dir.mkdir(parents=True, exist_ok=True)

    result_path = get_full_result_path(sample_name)

    with open(result_path, "wb") as f:
        pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)

    print("Full result saved to:", result_path)


def load_full_result(sample_name):
    """
    Load the full GraphGeneratorResult object for one sample.
    """
    result_path = get_full_result_path(sample_name)

    if not result_path.exists():
        raise FileNotFoundError(
            f"Full result file not found for sample '{sample_name}': {result_path}"
        )

    with open(result_path, "rb") as f:
        result = pickle.load(f)

    return result


# ------------------------------------------------------------
# ROI helpers
# ------------------------------------------------------------

def make_roi_from_mask(mask, pad, shape):
    """
    Create ROI bounds around a binary mask.

    Returns
    -------
    tuple
        (z0, z1, y0, y1, x0, x1)
    """
    if not np.any(mask):
        raise ValueError("Cannot build ROI from an empty mask.")

    zz, yy, xx = np.where(mask)

    z0 = max(int(zz.min()) - pad, 0)
    z1 = min(int(zz.max()) + pad + 1, shape[0])

    y0 = max(int(yy.min()) - pad, 0)
    y1 = min(int(yy.max()) + pad + 1, shape[1])

    x0 = max(int(xx.min()) - pad, 0)
    x1 = min(int(xx.max()) + pad + 1, shape[2])

    return z0, z1, y0, y1, x0, x1


def make_roi_from_points(points_zyx, pad, shape):
    """
    Create ROI bounds around a set of points in z-y-x order.
    """
    points = np.asarray(points_zyx, dtype=float)

    if points.ndim != 2 or points.shape[1] != 3 or len(points) == 0:
        raise ValueError("points_zyx must have shape (N, 3).")

    z0 = max(int(np.floor(points[:, 0].min())) - pad, 0)
    z1 = min(int(np.ceil(points[:, 0].max())) + pad + 1, shape[0])

    y0 = max(int(np.floor(points[:, 1].min())) - pad, 0)
    y1 = min(int(np.ceil(points[:, 1].max())) + pad + 1, shape[1])

    x0 = max(int(np.floor(points[:, 2].min())) - pad, 0)
    x1 = min(int(np.ceil(points[:, 2].max())) + pad + 1, shape[2])

    return z0, z1, y0, y1, x0, x1


def crop_array(arr, roi_bounds):
    """
    Crop a z-y-x array using ROI bounds.
    """
    z0, z1, y0, y1, x0, x1 = roi_bounds
    return arr[z0:z1, y0:y1, x0:x1]


# ------------------------------------------------------------
# Mesh helpers
# ------------------------------------------------------------

def add_mesh_to_viewer(
    viewer,
    mesh,
    name,
    origin_zyx=None,
    opacity=0.5,
    colormap="gray",
    visible=True,
):
    """
    Add a trimesh mesh to Napari as a surface layer.
    """
    if mesh is None:
        return None

    vertices = np.asarray(mesh.vertices, dtype=float)

    if origin_zyx is not None:
        vertices = vertices - np.asarray(origin_zyx, dtype=float)

    faces = np.asarray(mesh.faces, dtype=np.int64)
    values = np.ones(len(vertices), dtype=float)

    return viewer.add_surface(
        (vertices, faces, values),
        name=name,
        opacity=opacity,
        colormap=colormap,
        contrast_limits=(0, 1),
        visible=visible,
    )


def mesh_binary_roi(mask, pad=2):
    """
    Mesh one binary ROI mask in ROI-local coordinates.
    """
    mask = mask.astype(bool, copy=False)

    if not np.any(mask):
        return None

    return liver_graph_generator.mesh_binary_mask(mask, pad=pad)


def get_or_create_structure_mesh(sample_name, structure_name, mask, pad=2, force_recompute=False):
    """
    Create or load a cached mesh for a full structure.

    structure_name examples:
        bc
        sinusoids
        cv
        pv
    """
    cache_dir = get_visualization_cache_dir(sample_name)
    cache_dir.mkdir(parents=True, exist_ok=True)

    mesh_path = cache_dir / f"{structure_name}_mesh.pkl"

    if mesh_path.exists() and not force_recompute:
        with open(mesh_path, "rb") as f:
            mesh = pickle.load(f)

        print(f"Loaded cached {structure_name} mesh:", mesh_path)
        return mesh

    print(f"Meshing {structure_name}...")

    mesh = liver_graph_generator.mesh_binary_mask(mask, pad=pad)

    with open(mesh_path, "wb") as f:
        pickle.dump(mesh, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Saved {structure_name} mesh:", mesh_path)

    return mesh


# ------------------------------------------------------------
# Graph/vector visualization helpers
# ------------------------------------------------------------

def graph_edges_to_napari_vectors(edge_features, node_features):
    """
    Convert graph edges into Napari vector format.
    """
    centers_df = node_features.set_index("cell_id")[
        [
            "cell_centroid_z",
            "cell_centroid_y",
            "cell_centroid_x",
        ]
    ]

    vectors = []

    for row in edge_features.itertuples(index=False):
        cell_u = int(row.cell_u)
        cell_v = int(row.cell_v)

        if cell_u not in centers_df.index:
            continue

        if cell_v not in centers_df.index:
            continue

        p0 = centers_df.loc[cell_u].to_numpy(float)
        p1 = centers_df.loc[cell_v].to_numpy(float)

        direction = p1 - p0

        vectors.append([p0, direction])

    if len(vectors) == 0:
        return np.empty((0, 2, 3), dtype=float)

    return np.asarray(vectors, dtype=float)


def feature_axis_to_napari_vectors(
    node_features,
    axis_prefix,
    length=70.0,
    bidirectional=True,
):
    """
    Convert one vector feature into Napari vector format.
    """
    center_cols = [
        "cell_centroid_z",
        "cell_centroid_y",
        "cell_centroid_x",
    ]

    axis_cols = [
        f"{axis_prefix}_z",
        f"{axis_prefix}_y",
        f"{axis_prefix}_x",
    ]

    required_cols = center_cols + axis_cols

    if not all(col in node_features.columns for col in required_cols):
        return np.empty((0, 2, 3), dtype=float)

    df = node_features[
        ["cell_id"] + center_cols + axis_cols
    ].dropna(subset=required_cols).copy()

    if len(df) == 0:
        return np.empty((0, 2, 3), dtype=float)

    centers = df[center_cols].to_numpy(float)
    axes = df[axis_cols].to_numpy(float)

    norms = np.linalg.norm(axes, axis=1)
    valid = norms > 1e-12

    centers = centers[valid]
    axes = axes[valid] / norms[valid, None]

    if len(centers) == 0:
        return np.empty((0, 2, 3), dtype=float)

    if bidirectional:
        starts = centers - 0.5 * length * axes
        directions = length * axes
    else:
        starts = centers
        directions = length * axes

    return np.stack([starts, directions], axis=1)


def get_vector_feature_names(result, include_centroid=False):
    """
    Return vector feature names registered in the result.
    """
    vector_features = result.feature_registry.get(level="node", kind="vector")

    if include_centroid:
        return vector_features

    return [
        feature
        for feature in vector_features
        if feature.startswith("axis_")
    ]

In [1]:
# ============================================================
# BLOCK 3 — Functions to run graph generation for all samples
# ============================================================

def build_graph_config(output_dir, compute_reference_frame):
    """
    Build the graph generator configuration.
    """
    config = GraphGeneratorConfig(
        # Output
        output_dir=output_dir,
        save_outputs=True,
        save_graph_pickle=True,
        prefer_parquet=True,

        # Raw labels in CV/PV image
        raw_id_cv=100,
        raw_id_pv=200,

        # Internal special labels
        id_cv=65531,
        id_pv=65532,
        id_border=65533,
        id_bile=65534,
        id_sinu=65535,

        # Nuclear labels
        nucleus_type_labels={
            100: "normal_hepatocyte_nucleus",
            200: "metaphase",
            300: "anaphase_telophase",
            400: "cytokinesis",
        },

        nucleus_feature_prefix={
            100: "normal",
            200: "metaphase",
            300: "anaphase_telophase",
            400: "cytokinesis",
        },

        # Small-nucleus pre-filtering
        filter_small_nuclei=True,
        min_nucleus_voxels=50,
        min_nucleus_voxels_by_type={},

        # Cell-label expansion
        expansion_dist=20,

        # Mesh generation
        cell_mesh_pad=2,
        nucleus_mesh_pad=2,
        vessel_mesh_pad=2,
        n_jobs=8,

        # Face-wise surface probing
        probe_start_dist=1.0,
        probe_end_dist=30.0,
        probe_step_dist=0.5,
        border_margin=5,
        exclude_border_cells=True,

        # Diagnostics
        collect_void_face_diagnostics=True,
        max_void_faces_per_cell=500,

        # Physics
        compute_reference_frame=compute_reference_frame,
        compute_coarse_graining=True,
        compute_pairwise_cosines=True,
        coarse_sigma_vox=80.0,

        verbose=True,
    )

    return config


def sample_has_cv_and_pv(sample_name):
    """
    Check whether the sample contains both CV and PV labels.
    """
    arrays = open_sample_memmaps(sample_name)

    vcvp = arrays["vcvp"]

    n_cv = np.count_nonzero(vcvp == 100)
    n_pv = np.count_nonzero(vcvp == 200)

    del arrays
    gc.collect()

    return (n_cv > 0) and (n_pv > 0)


def run_one_sample(sample_name, overwrite=False):
    """
    Run graph generation for one sample and export all outputs.
    """
    output_dir = get_sample_output_dir(sample_name)
    result_path = get_full_result_path(sample_name)

    if result_path.exists() and not overwrite:
        print(f"Skipping {sample_name}. Full result already exists:")
        print(result_path)
        return

    sample_dir = get_sample_dir(sample_name)
    output_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print("Running sample:", sample_name)
    print("Input folder:", sample_dir)
    print("Output folder:", output_dir)
    print("=" * 80)

    inspect_sample_files(sample_name)

    compute_reference_frame = sample_has_cv_and_pv(sample_name)

    print("Compute reference frame:", compute_reference_frame)

    config = build_graph_config(
        output_dir=output_dir,
        compute_reference_frame=compute_reference_frame,
    )

    generator = LiverGraphGenerator(config)

    result = generator.run_from_paths(
        data_dir=sample_dir,
        cells_file=EXPECTED_FILES["cells"],
        bc_file=EXPECTED_FILES["bc"],
        sinusoids_file=EXPECTED_FILES["sinusoids"],
        nuclei_file=EXPECTED_FILES["nuclei"],
        vcvp_file=EXPECTED_FILES["vcvp"],
        crop_bounds=None,
        sample_id=sample_name,
        output_dir=output_dir,
    )

    save_full_result(result, sample_name)

    print("Completed sample:", sample_name)
    print("Nodes:", result.graph.number_of_nodes())
    print("Edges:", result.graph.number_of_edges())
    print("Node features:", result.node_features.shape)
    print("Edge features:", result.edge_features.shape)

    del result
    del generator
    gc.collect()


def run_all_samples(sample_names, overwrite=False):
    """
    Run graph generation for all samples.
    """
    for sample_name in sample_names:
        run_one_sample(sample_name, overwrite=overwrite)

    print("All samples completed.")

In [ ]:
# ============================================================
# BLOCK 4 — Diagnostic visualization functions
# ============================================================

def visualize_cells_with_more_than_two_nuclei(sample_name, pad=40):
    """
    Visualize cells with more than two assigned nuclei.
    """
    result = load_full_result(sample_name)
    arrays = open_sample_memmaps(sample_name)

    cells = arrays["cells"]
    nuclei = arrays["nuclei"]

    gt2_cells_df = result.diagnostics["cells_with_more_than_two_nuclei"].copy()
    assigned_nuclei_df = result.diagnostics["assigned_nuclei"].copy()

    display(gt2_cells_df)

    if len(gt2_cells_df) == 0:
        print("No cells with more than two nuclei were found.")
        return None

    def parse_assigned_nucleus_ids(value):
        if pd.isna(value):
            return []
        value = str(value).strip()
        if value == "":
            return []
        return [int(x) for x in value.split(";") if x.strip() != ""]

    gt2_cell_ids = gt2_cells_df["cell_id"].astype(int).tolist()

    gt2_nucleus_ids = []
    for value in gt2_cells_df["assigned_nucleus_ids"]:
        gt2_nucleus_ids.extend(parse_assigned_nucleus_ids(value))
    gt2_nucleus_ids = sorted(set(gt2_nucleus_ids))

    print("Cells with more than two nuclei:", gt2_cell_ids)
    print("Assigned nucleus IDs:", gt2_nucleus_ids)

    gt2_assigned_nuclei_df = assigned_nuclei_df[
        assigned_nuclei_df["owner_cell_id"].astype(int).isin(gt2_cell_ids)
    ].copy()

    display(gt2_assigned_nuclei_df)

    cell_mask = np.isin(cells, gt2_cell_ids)
    roi_bounds = make_roi_from_mask(cell_mask, pad=pad, shape=cells.shape)
    roi_origin = np.array([roi_bounds[0], roi_bounds[2], roi_bounds[4]], dtype=float)

    print("ROI bounds:", roi_bounds)

    cells_roi = crop_array(cells, roi_bounds)
    nuclei_roi = crop_array(nuclei, roi_bounds)

    selected_cells_roi = np.where(
        np.isin(cells_roi, gt2_cell_ids),
        cells_roi,
        0,
    ).astype(np.uint16)

    cell_point_rows = []
    for cell_id in gt2_cell_ids:
        if cell_id not in result.cell_meshes:
            continue

        centroid = np.asarray(result.cell_meshes[cell_id].centroid, dtype=float)
        centroid_local = centroid - roi_origin

        cell_point_rows.append({
            "cell_id": int(cell_id),
            "z": centroid_local[0],
            "y": centroid_local[1],
            "x": centroid_local[2],
            "label": f"cell {cell_id}",
        })

    cell_points_df = pd.DataFrame(cell_point_rows)

    nucleus_point_rows = []
    for nucleus_id in gt2_nucleus_ids:
        if nucleus_id not in result.nucleus_data:
            continue

        nuc = result.nucleus_data[nucleus_id]
        centroid = np.asarray(nuc["mesh_centroid_zyx"], dtype=float)
        centroid_local = centroid - roi_origin

        owner_row = gt2_assigned_nuclei_df[
            gt2_assigned_nuclei_df["nucleus_id"].astype(int) == nucleus_id
        ]

        owner_cell_id = int(owner_row["owner_cell_id"].iloc[0]) if len(owner_row) > 0 else -1

        nucleus_point_rows.append({
            "nucleus_id": int(nucleus_id),
            "owner_cell_id": owner_cell_id,
            "type_label": int(nuc["type_label"]),
            "type_name": str(nuc["type_name"]),
            "z": centroid_local[0],
            "y": centroid_local[1],
            "x": centroid_local[2],
            "label": f"nuc {nucleus_id}\ncell {owner_cell_id}\n{nuc['type_name']}",
        })

    nucleus_points_df = pd.DataFrame(nucleus_point_rows)

    display(cell_points_df)
    display(nucleus_points_df)

    viewer = napari.Viewer(ndisplay=3)
    viewer.theme = "dark"

    viewer.add_labels(
        selected_cells_roi,
        name="Cells with >2 nuclei",
        opacity=0.45,
        visible=True,
    )

    viewer.add_labels(
        nuclei_roi.astype(np.uint16),
        name="Original nucleus type labels in ROI",
        opacity=0.35,
        visible=False,
    )

    if len(cell_points_df) > 0:
        viewer.add_points(
            cell_points_df[["z", "y", "x"]].to_numpy(float),
            name="Cell ID labels",
            size=12,
            face_color="white",
            border_color="black",
            opacity=1.0,
            features=cell_points_df,
            text={
                "string": "{label}",
                "size": 10,
                "color": "white",
                "visible": True,
            },
        )

    if len(nucleus_points_df) > 0:
        viewer.add_points(
            nucleus_points_df[["z", "y", "x"]].to_numpy(float),
            name="Assigned nucleus ID labels",
            size=8,
            face_color="cyan",
            border_color="black",
            opacity=1.0,
            features=nucleus_points_df,
            text={
                "string": "{label}",
                "size": 8,
                "color": "cyan",
                "visible": True,
            },
        )

    for nucleus_id in gt2_nucleus_ids:
        if nucleus_id not in result.nucleus_meshes:
            continue

        nuc_mesh = result.nucleus_meshes[nucleus_id]
        nuc_data = result.nucleus_data[nucleus_id]

        add_mesh_to_viewer(
            viewer,
            nuc_mesh,
            name=f"Nucleus mesh {nucleus_id} — {nuc_data['type_name']}",
            origin_zyx=roi_origin,
            opacity=0.75,
            colormap="magma",
            visible=True,
        )

    viewer.dims.ndisplay = 3
    viewer.camera.angles = (65, 35, 120)
    viewer.camera.zoom = 1.2

    print("Napari viewer created for cells with more than two nuclei.")
    return viewer


def visualize_unassigned_nuclei(sample_name, pad=60):
    """
    Visualize nuclei that were not assigned to any hepatocyte.
    """
    result = load_full_result(sample_name)
    arrays = open_sample_memmaps(sample_name)

    cells = arrays["cells"]
    nuclei = arrays["nuclei"]

    unassigned_df = result.diagnostics["unassigned_nuclei"].copy()

    display(unassigned_df)

    if len(unassigned_df) == 0:
        print("No unassigned nuclei were found.")
        return None

    nucleus_ids = sorted(unassigned_df["nucleus_id"].astype(int).tolist())

    nucleus_point_rows = []

    for nucleus_id in nucleus_ids:
        if nucleus_id not in result.nucleus_data:
            continue

        nuc = result.nucleus_data[nucleus_id]
        centroid = np.asarray(nuc["mesh_centroid_zyx"], dtype=float)

        row_match = unassigned_df[
            unassigned_df["nucleus_id"].astype(int) == nucleus_id
        ]

        reason = str(row_match["reason"].iloc[0]) if "reason" in row_match.columns else "unassigned"

        nucleus_point_rows.append({
            "nucleus_id": int(nucleus_id),
            "type_label": int(nuc["type_label"]),
            "type_name": str(nuc["type_name"]),
            "reason": reason,
            "z": centroid[0],
            "y": centroid[1],
            "x": centroid[2],
            "label": f"nuc {nucleus_id}\n{nuc['type_name']}\n{reason}",
        })

    nucleus_points_df = pd.DataFrame(nucleus_point_rows)

    if len(nucleus_points_df) == 0:
        print("No nucleus points could be built.")
        return None

    display(nucleus_points_df)

    points = nucleus_points_df[["z", "y", "x"]].to_numpy(float)
    roi_bounds = make_roi_from_points(points, pad=pad, shape=cells.shape)
    roi_origin = np.array([roi_bounds[0], roi_bounds[2], roi_bounds[4]], dtype=float)

    print("ROI bounds:", roi_bounds)

    cells_roi = crop_array(cells, roi_bounds)
    nuclei_roi = crop_array(nuclei, roi_bounds)

    nucleus_points_local_df = nucleus_points_df.copy()
    nucleus_points_local_df["z"] -= roi_origin[0]
    nucleus_points_local_df["y"] -= roi_origin[1]
    nucleus_points_local_df["x"] -= roi_origin[2]

    viewer = napari.Viewer(ndisplay=3)
    viewer.theme = "dark"

    viewer.add_labels(
        cells_roi.astype(np.uint16),
        name="Hepatocyte labels in area",
        opacity=0.35,
        visible=True,
    )

    viewer.add_labels(
        nuclei_roi.astype(np.uint16),
        name="Original nucleus type labels in ROI",
        opacity=0.25,
        visible=False,
    )

    viewer.add_points(
        nucleus_points_local_df[["z", "y", "x"]].to_numpy(float),
        name="Unassigned nucleus centroids",
        size=10,
        face_color="red",
        border_color="white",
        opacity=1.0,
        features=nucleus_points_local_df,
        text={
            "string": "{label}",
            "size": 9,
            "color": "red",
            "visible": True,
        },
    )

    for nucleus_id in nucleus_ids:
        if nucleus_id not in result.nucleus_meshes:
            continue

        nuc_mesh = result.nucleus_meshes[nucleus_id]
        nuc_data = result.nucleus_data[nucleus_id]

        add_mesh_to_viewer(
            viewer,
            nuc_mesh,
            name=f"Nucleus mesh {nucleus_id} — {nuc_data['type_name']}",
            origin_zyx=roi_origin,
            opacity=0.85,
            colormap="magma",
            visible=True,
        )

    viewer.dims.ndisplay = 3
    viewer.camera.angles = (65, 35, 120)
    viewer.camera.zoom = 1.2

    print("Napari viewer created for unassigned nuclei.")
    return viewer


def visualize_void_faces(sample_name, top_n_void_cells=6, pad=60, max_void_faces_per_cell=3000):
    """
    Visualize cells with the largest void surface area.
    """
    result = load_full_result(sample_name)
    arrays = open_sample_memmaps(sample_name)

    cells = arrays["cells"]
    bc = arrays["bc"]
    sinusoids = arrays["sinusoids"]

    graph_node_ids = set(result.node_features["cell_id"].astype(int).tolist())

    void_rows = []

    for cell_id, data in result.cell_data.items():
        cell_id = int(cell_id)

        if cell_id not in graph_node_ids:
            continue

        face_labels = np.asarray(data["face_labels"])
        face_areas = np.asarray(data["face_areas"], dtype=float)

        void_mask = face_labels == 0

        n_void_faces = int(np.count_nonzero(void_mask))
        area_void = float(np.sum(face_areas[void_mask]))
        area_total = float(np.sum(face_areas))

        if n_void_faces == 0:
            continue

        void_rows.append({
            "cell_id": cell_id,
            "n_void_faces": n_void_faces,
            "area_void": area_void,
            "cell_surface_area": area_total,
            "frac_void": area_void / area_total if area_total > 0 else np.nan,
        })

    void_summary_df = pd.DataFrame(void_rows)

    if len(void_summary_df) == 0:
        print("No void faces found.")
        return None

    void_summary_df = void_summary_df.sort_values(
        ["area_void", "n_void_faces"],
        ascending=[False, False],
    ).reset_index(drop=True)

    display(void_summary_df.head(20))

    selected_cells = (
        void_summary_df
        .head(top_n_void_cells)["cell_id"]
        .astype(int)
        .tolist()
    )

    print("Selected cells:", selected_cells)

    selected_mask = np.isin(cells, selected_cells)
    roi_bounds = make_roi_from_mask(selected_mask, pad=pad, shape=cells.shape)
    roi_origin = np.array([roi_bounds[0], roi_bounds[2], roi_bounds[4]], dtype=float)

    print("ROI bounds:", roi_bounds)

    cells_roi = crop_array(cells, roi_bounds)
    bc_roi = crop_array(bc, roi_bounds)
    sinusoids_roi = crop_array(sinusoids, roi_bounds)

    selected_cells_roi = np.where(
        np.isin(cells_roi, selected_cells),
        cells_roi,
        0,
    ).astype(np.uint16)

    print("Meshing BC ROI...")
    bc_mesh = mesh_binary_roi(bc_roi > 0, pad=2)

    print("Meshing sinusoid ROI...")
    sin_mesh = mesh_binary_roi(sinusoids_roi > 0, pad=2)

    viewer = napari.Viewer(ndisplay=3)
    viewer.theme = "dark"

    viewer.add_labels(
        selected_cells_roi,
        name="Selected hepatocyte labels",
        opacity=0.35,
        visible=True,
    )

    viewer.add_labels(
        cells_roi.astype(np.uint16),
        name="All hepatocyte labels in ROI",
        opacity=0.08,
        visible=False,
    )

    add_mesh_to_viewer(
        viewer,
        bc_mesh,
        name="BC mesh in ROI",
        opacity=0.65,
        colormap="green",
        visible=True,
    )

    add_mesh_to_viewer(
        viewer,
        sin_mesh,
        name="Sinusoid mesh in ROI",
        opacity=0.35,
        colormap="cyan",
        visible=True,
    )

    cell_point_rows = []

    for cell_id in selected_cells:
        mesh = result.cell_meshes[cell_id]
        data = result.cell_data[cell_id]

        centroid_local = np.asarray(mesh.centroid, dtype=float) - roi_origin
        row = void_summary_df[void_summary_df["cell_id"] == cell_id].iloc[0]

        cell_point_rows.append({
            "cell_id": int(cell_id),
            "frac_void": float(row["frac_void"]),
            "z": centroid_local[0],
            "y": centroid_local[1],
            "x": centroid_local[2],
            "label": f"cell {cell_id}\nvoid {row['frac_void']:.3f}",
        })

        vertices_local = np.asarray(mesh.vertices, dtype=float) - roi_origin
        faces = np.asarray(mesh.faces, dtype=np.int64)

        face_labels = np.asarray(data["face_labels"])
        void_face_indices = np.where(face_labels == 0)[0]

        if len(void_face_indices) > max_void_faces_per_cell:
            void_face_indices = void_face_indices[:max_void_faces_per_cell]

        values = np.ones(len(vertices_local), dtype=float)

        viewer.add_surface(
            (vertices_local, faces, values),
            name=f"Full cell mesh {cell_id}",
            opacity=0.10,
            colormap="gray",
            contrast_limits=(0, 1),
            visible=False,
        )

        if len(void_face_indices) > 0:
            viewer.add_surface(
                (vertices_local, faces[void_face_indices], values),
                name=f"VOID faces cell {cell_id}",
                opacity=0.95,
                colormap="magma",
                contrast_limits=(0, 1),
                visible=True,
            )

    cell_points_df = pd.DataFrame(cell_point_rows)

    if len(cell_points_df) > 0:
        viewer.add_points(
            cell_points_df[["z", "y", "x"]].to_numpy(float),
            name="Void-cell centroids",
            size=10,
            face_color="white",
            border_color="black",
            opacity=1.0,
            features=cell_points_df,
            text={
                "string": "{label}",
                "size": 8,
                "color": "white",
                "visible": True,
            },
        )

    viewer.dims.ndisplay = 3
    viewer.camera.angles = (65, 35, 120)
    viewer.camera.zoom = 1.0

    print("Napari viewer created for void-face diagnostics.")
    return viewer

In [ ]:
# ============================================================
# BLOCK 5 — Global graph visualization
# ============================================================

def visualize_graph(
    sample_name,
    vector_features=None,
    show_all_vectors=False,
    vector_length=70.0,
    cg_vector_length=90.0,
    mesh_structures=True,
    force_recompute_structure_meshes=False,
):
    """
    Visualize one graph globally in Napari.

    Parameters
    ----------
    sample_name : str
        Sample name.

    vector_features : list or None
        Vector feature names to display.
        If None, all vector feature layers are added but hidden.

    show_all_vectors : bool
        If True, all vector layers are visible by default.

    mesh_structures : bool
        If True, mesh BC, sinusoids, CV and PV.

    force_recompute_structure_meshes : bool
        If True, recompute cached structure meshes.
    """
    result = load_full_result(sample_name)
    arrays = open_sample_memmaps(sample_name)

    node_features = result.node_features.copy()
    edge_features = result.edge_features.copy()

    viewer = napari.Viewer(ndisplay=3)
    viewer.theme = "dark"

    # --------------------------------------------------------
    # Add anatomical structure meshes
    # --------------------------------------------------------

    if mesh_structures:
        bc_mask = arrays["bc"] > 0
        sin_mask = arrays["sinusoids"] > 0
        cv_mask = arrays["vcvp"] == 100
        pv_mask = arrays["vcvp"] == 200

        bc_mesh = get_or_create_structure_mesh(
            sample_name,
            "bc",
            bc_mask,
            pad=2,
            force_recompute=force_recompute_structure_meshes,
        )

        sin_mesh = get_or_create_structure_mesh(
            sample_name,
            "sinusoids",
            sin_mask,
            pad=2,
            force_recompute=force_recompute_structure_meshes,
        )

        cv_mesh = get_or_create_structure_mesh(
            sample_name,
            "cv",
            cv_mask,
            pad=2,
            force_recompute=force_recompute_structure_meshes,
        )

        pv_mesh = get_or_create_structure_mesh(
            sample_name,
            "pv",
            pv_mask,
            pad=2,
            force_recompute=force_recompute_structure_meshes,
        )

        add_mesh_to_viewer(
            viewer,
            bc_mesh,
            name="BC mesh",
            opacity=0.50,
            colormap="green",
            visible=True,
        )

        add_mesh_to_viewer(
            viewer,
            sin_mesh,
            name="Sinusoids mesh",
            opacity=0.30,
            colormap="cyan",
            visible=True,
        )

        add_mesh_to_viewer(
            viewer,
            cv_mesh,
            name="CV mesh",
            opacity=0.80,
            colormap="blue",
            visible=True,
        )

        add_mesh_to_viewer(
            viewer,
            pv_mesh,
            name="PV mesh",
            opacity=0.80,
            colormap="orange",
            visible=True,
        )

    # --------------------------------------------------------
    # Add graph nodes
    # --------------------------------------------------------

    node_points = node_features[
        [
            "cell_centroid_z",
            "cell_centroid_y",
            "cell_centroid_x",
        ]
    ].to_numpy(float)

    viewer.add_points(
        node_points,
        name="Graph nodes",
        size=7,
        face_color="white",
        border_color="white",
        opacity=1.0,
        features=node_features[["cell_id"]],
        text={
            "string": "{cell_id}",
            "size": 7,
            "color": "white",
            "visible": False,
        },
    )

    # --------------------------------------------------------
    # Add graph edges
    # --------------------------------------------------------

    edge_vectors = graph_edges_to_napari_vectors(
        edge_features=edge_features,
        node_features=node_features,
    )

    viewer.add_vectors(
        edge_vectors,
        name="Graph edges",
        edge_color="yellow",
        edge_width=2.0,
        length=1.0,
        opacity=0.85,
    )

    # --------------------------------------------------------
    # Add vector features
    # --------------------------------------------------------

    all_axis_features = get_vector_feature_names(result, include_centroid=False)

    if vector_features is None:
        vector_features_to_add = all_axis_features
    else:
        vector_features_to_add = vector_features

    print("Vector features available:")
    for name in all_axis_features:
        print(" -", name)

    print("\nVector features added:")
    for name in vector_features_to_add:
        print(" -", name)

    for axis_name in vector_features_to_add:
        is_cg = axis_name.startswith("axis_cg_")
        length = cg_vector_length if is_cg else vector_length

        vectors = feature_axis_to_napari_vectors(
            node_features=node_features,
            axis_prefix=axis_name,
            length=length,
            bidirectional=True,
        )

        if len(vectors) == 0:
            print("Skipping empty vector feature:", axis_name)
            continue

        viewer.add_vectors(
            vectors,
            name=f"Vector — {axis_name}",
            edge_width=2.5 if is_cg else 2.0,
            length=1.0,
            opacity=0.95,
            visible=show_all_vectors,
        )

    viewer.dims.ndisplay = 3
    viewer.camera.angles = (65, 35, 120)
    viewer.camera.zoom = 0.8

    print("Napari graph viewer created.")
    print("Sample:", sample_name)
    print("Nodes:", len(node_points))
    print("Edges:", len(edge_vectors))

    return viewer

In [ ]:
# ============================================================
# BLOCK 6 — User settings
# ============================================================

# ------------------------------------------------------------
# Base folder shared by all sample folders
# ------------------------------------------------------------

BASE_DATA_DIR = Path.home() / "Desktop/Master/thesis/Liver_Tissue/Regeneration"

# ------------------------------------------------------------
# Output folder for all generated graphs
# ------------------------------------------------------------

GRAPH_OUTPUT_DIR = BASE_DATA_DIR / "graphs"

GRAPH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Sample folder names
# ------------------------------------------------------------
# Each sample folder must contain:
#   Cells.tiff
#   Nucleos.tiff
#   BC.tiff
#   Sinusoids.tiff
#   CV-PV.tiff

SAMPLE_FOLDER_NAMES = [
    "sample_01",
    "sample_02",
    "sample_03",
]

print("Base data folder:", BASE_DATA_DIR)
print("Graph output folder:", GRAPH_OUTPUT_DIR)
print("Samples:")
for sample_name in SAMPLE_FOLDER_NAMES:
    print(" -", sample_name)

In [ ]:
# ============================================================
# BLOCK 7 — Inspect all sample files before running
# ============================================================

for sample_name in SAMPLE_FOLDER_NAMES:
    print("=" * 80)
    print("Inspecting sample:", sample_name)
    print("=" * 80)
    inspect_sample_files(sample_name)

In [ ]:
# ============================================================
# BLOCK 8 — Run graph generation for all samples
# ============================================================

run_all_samples(
    SAMPLE_FOLDER_NAMES,
    overwrite=False,
)

In [ ]:
# ============================================================
# BLOCK 9 — Diagnostic: cells with more than two nuclei
# ============================================================

sample_name = "sample_01"

viewer_gt2 = visualize_cells_with_more_than_two_nuclei(
    sample_name,
    pad=40,
)

In [ ]:
# ============================================================
# BLOCK 10 — Diagnostic: unassigned nuclei
# ============================================================

sample_name = "sample_01"

viewer_unassigned = visualize_unassigned_nuclei(
    sample_name,
    pad=60,
)

In [ ]:
# ============================================================
# BLOCK 11 — Diagnostic: void faces
# ============================================================

sample_name = "sample_01"

viewer_void = visualize_void_faces(
    sample_name,
    top_n_void_cells=6,
    pad=60,
    max_void_faces_per_cell=3000,
)

In [ ]:
# ============================================================
# BLOCK 12 — Visualize full graph
# ============================================================

sample_name = "sample_01"

viewer_graph = visualize_graph(
    sample_name,
    vector_features=None,
    show_all_vectors=False,
    vector_length=70.0,
    cg_vector_length=90.0,
    mesh_structures=True,
    force_recompute_structure_meshes=False,
)

In [ ]:
# ============================================================
# BLOCK 13 — Visualize graph with selected vector features
# ============================================================

sample_name = "sample_01"

viewer_graph_selected = visualize_graph(
    sample_name,
    vector_features=[
        "axis_apical_bipolar",
        "axis_basal_bipolar",
        "axis_inertia_vec3",
        "axis_nucleus_inertia_vec3",
    ],
    show_all_vectors=True,
    vector_length=70.0,
    cg_vector_length=90.0,
    mesh_structures=True,
    force_recompute_structure_meshes=False,
)

In [ ]:
# ============================================================
# BLOCK 14 — Inspect scalar and vector features
# ============================================================

sample_name = "sample_01"

result = load_full_result(sample_name)

scalar_features = result.feature_registry.get(level="node", kind="scalar")
vector_features = result.feature_registry.get(level="node", kind="vector")

print("=" * 80)
print("SCALAR NODE FEATURES")
print("=" * 80)
print("Number of scalar features:", len(scalar_features))
print()

for i, feature_name in enumerate(scalar_features, start=1):
    print(f"{i:03d}. {feature_name}")

print("\n" + "=" * 80)
print("VECTOR NODE FEATURES")
print("=" * 80)
print("Number of vector features:", len(vector_features))
print()

for i, feature_name in enumerate(vector_features, start=1):
    print(f"{i:03d}. {feature_name}")